# Timedep water level application

In a fixed water level calculation, the load is a single water level. A real water level changes over time, and a pipe grows while it is loaded. The class [ProbPipingTimeDepWaterlevel](/reference/ProbPipingTimeDepWaterlevel.html) calculates the failure probability for a water level _series_: the pipe grows with a chosen growth model, and the limit state is evaluated at the end of the series. In this notebook, all water level series are synthetic.

The package calculates one series per call (`timedep_waterlevel_failureprobability()`). You decide how to loop over series, average over realisations or integrate over frequencies.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from probabilistic_piping import (
    pipe_growth,
    ProbInput,
    ProbPipingTimeDepWaterlevel,
)

## Input
The same input data as in the introduction is used, but a timedep calculation needs a few extra rows for the pipe growth. We add them in code as `geen stochast` rows (a fixed value), except for the growth coefficient `dldt_Ce`, which is a lognormal stochast:

- `dldt_modeltype`: the pipe growth model, `constant`, `Pol2019` or `Pol2022`. Here we use `Pol2022`, which needs `dldt_Ce`. The model is based on Pol (2022), equation 6.5, and Pol et al. (2024); the references are in the documentation of [pipe_growth](/reference/pipe_growth.html).
- `l_ini`: the initial pipe length as a fraction of the seepage length `L`. Here 0.
- `detect_true`: whether the pipe is detected, so that an intervention follows. Here `False`, we switch it on later.
- `dldt_Ce`: the growth coefficient of `Pol2022`.

In [ ]:
def add_row(df, name, kind, value, stdev=None):
    """Add an input row, a fixed value or a stochast with a standard deviation."""
    df.loc[name] = np.nan
    df.loc[name, ["Beschrijving", "Waarde", "Kansverdeling"]] = [name, value, kind]
    if stdev is not None:
        df.loc[name, ["Mean", "StDev", "Spreiding"]] = [value, stdev, stdev]
        df.loc[name, ["Spreidingstype", "Verschuiving"]] = ["Standaardafwijking", 0.0]


data_path = Path("../../../test/data/full_test.xlsx")
df_input = pd.read_excel(data_path, sheet_name="input", index_col=0, header=0)
add_row(df_input, "dldt_modeltype", "geen stochast", "Pol2022")
add_row(df_input, "l_ini", "geen stochast", 0.0)
add_row(df_input, "detect_true", "geen stochast", False)
add_row(df_input, "dldt_Ce", "Lognormaal", 0.1, stdev=0.05)

## Series loop
We make three synthetic triangle hydrographs: the water level rises from 0 m to a peak in 30 h and falls back in 30 h, with a 1 h step. They differ in their peak. For each series we calculate the failure probability with FORM Abdo-Rackwitz and collect the results.

In [ ]:
def hydrograph(peak, dt=1.0):
    """Triangle hydrograph [h, m+NAP] over 60 h with the given peak."""
    t = np.arange(0, 60 + dt / 2, dt)
    return t, np.interp(t, [0, 30, 60], [0.0, peak, 0.0])


def timedep_form(df, peak, dt=None):
    """Timedep calculation with FORM Abdo-Rackwitz for one triangle series."""
    inp = ProbInput.from_dataframe(df)
    inp.calc_options["Rekentechniek"] = "FORM AbdoRackwitz"
    t, h = hydrograph(peak)
    prob = ProbPipingTimeDepWaterlevel(progress=False)
    return prob.timedep_waterlevel_failureprobability(inp, t, h, dt=dt)


peaks = [12.0, 13.0, 14.0]
results = [timedep_form(df_input, peak)[1] for peak in peaks]
pd.DataFrame(
    {
        "peak [m+NAP]": peaks,
        "failure probability": [r.prob_cond for r in results],
        "converged": [r.converged for r in results],
    }
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(peaks, [r.prob_cond for r in results], marker="o")
ax.set_xlabel("Peak water level [m+NAP]")
ax.set_ylabel("Failure probability")
ax.grid(True);

## Detection
With `detect_true` on, a pipe can be detected and an intervention stops the growth. The intervention takes place `intervention_hours` after the first erosion. Here, `intervention_hours` is fixed at 24 h (it may also be a continuous stochast). We compare the highest series with detection off and on.

In [ ]:
df_detect = df_input.copy()
df_detect.loc["detect_true", "Waarde"] = True
add_row(df_detect, "intervention_hours", "geen stochast", 24.0)

p_nodetect = timedep_form(df_input, 14.0)[1].prob_cond
p_detect = timedep_form(df_detect, 14.0)[1].prob_cond
pd.Series(
    {"detection off": p_nodetect, "detection on (24 h)": p_detect},
    name="failure probability",
).to_frame()

## Pipe growth at the design point
The calculation returns the settings and a result with the design point (`physical_design`). We set the design point on the settings and call [pipe_growth](/reference/pipe_growth.html), which returns the pipe length over time (`l_time`) for this single sample. The plot shows the pipe length together with the water level on a second axis.

In [ ]:
settings, result = timedep_form(df_input, 14.0)
settings.set_params_fromdict(result.physical_design)
growth = pipe_growth(settings)

t, h = settings.timegrid.ravel(), settings.h_timeseries.ravel()
fig, ax1 = plt.subplots(figsize=(6, 3))
ax1.plot(t, growth.l_time[:, 0], color="C0", label="pipe length")
ax1.axhline(settings.L, color="C0", ls=":", label="seepage length L")
ax1.set_xlabel("Time [h]")
ax1.set_ylabel("Pipe length [m]", color="C0")
ax1.legend(loc="upper left")
ax2 = ax1.twinx()
ax2.plot(t, h, color="C1")
ax2.set_ylabel("Water level [m+NAP]", color="C1");

## Time step
With the argument `dt`, the series is resampled to a fixed time step. The pipe grows per time step, so the result can depend on `dt`. **Always check that your result has converged on the time step.** A half-hourly step (`dt=0.5`) is what has been checked so far. Here the same series is calculated with three time steps.

In [ ]:
dts = [1.0, 0.5, 0.25]
res_dt = [timedep_form(df_input, 14.0, dt=dt)[1] for dt in dts]
pd.DataFrame(
    {
        "dt [h]": dts,
        "failure probability": [r.prob_cond for r in res_dt],
        "converged": [r.converged for r in res_dt],
    }
)

## Solvers
Use FORM Abdo-Rackwitz or Monte Carlo for timedep calculations. FORM Cobyla often does not converge on `Pol2022`, and directional sampling is still under investigation.